In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from catboost import CatBoostRegressor
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

In [ ]:
data = pd.read_excel('../../dataset/all data.xlsx',sheet_name='new')

In [12]:
group_reference = data['Reference'].copy()
group_species   = data['Species'].copy()
group_pfas      = data['PFAS'].copy()

In [13]:
data['logIpc'] = np.log10(data['Ipc'])
data['logSRC'] = np.log10(data['Soil residual concentration'])
data['logED'] = np.log10(data['ED'])

In [14]:
data.drop(columns=['Reference', 'Sample','Soil source','ES','PFAS','PFAS class','Species','Plant class','group','Ipc','Original tissue',
                   'ED','Soil residual concentration'], inplace=True)
X = data.drop(columns=['lnBCF'], errors='ignore').copy()
y = data['lnBCF'].copy()

In [6]:
cat_cols = [
    'Tissue',
    'Family','Genus','Order','monocot','woody','herb','crop','vegetable','legume','grass','perennial','edible'
]
cat_cols = [c for c in cat_cols if c in X.columns] 

In [7]:
best_params = {
    'iterations': 1803,
    'depth': 7,
    'learning_rate': 0.03823231749308767,
    'l2_leaf_reg': 7.85887570845998,
    'bagging_temperature': 0.8606437364021795,
    'random_strength': 8.94247879276211,
    'subsample': 0.8029919474739348,
    'colsample_bylevel': 0.7100100060503998,

    'loss_function': 'RMSE',
    'eval_metric': 'RMSE',
    'verbose': 0,
    'random_seed': 42
}

In [ ]:
# ============================================================
# Universal Group Validation Function
# ============================================================
def run_group_cv(groups, name='LOSO', min_n=5):

    # --------------------------------------------------------
    # storage
    # --------------------------------------------------------
    results = []

    all_true = []
    all_pred = []
    all_group = []

    unique_groups = pd.Series(groups).dropna().unique()

    print(f'\n========== Running {name} ==========')
    print(f'Total groups: {len(unique_groups)}')

    # ========================================================
    # Loop over groups
    # ========================================================
    for g in unique_groups:

        train_idx = (groups != g)
        test_idx  = (groups == g)

        X_train = X.loc[train_idx].copy()
        X_test  = X.loc[test_idx].copy()

        y_train = y.loc[train_idx].copy()
        y_test  = y.loc[test_idx].copy()

        # skip tiny test groups
        if len(y_test) < min_n:
            print(f'Skip {g} (n={len(y_test)})')
            continue

        # ----------------------------------------------------
        # model
        # ----------------------------------------------------
        model = CatBoostRegressor(**best_params)

        model.fit(
            X_train,
            y_train,
            cat_features=cat_cols,
            verbose=False
        )

        pred = model.predict(X_test)

        # ----------------------------------------------------
        # metrics
        # ----------------------------------------------------
        r2   = r2_score(y_test, pred)
        rmse = np.sqrt(mean_squared_error(y_test, pred))
        mae  = mean_absolute_error(y_test, pred)

        results.append({
            'Group': g,
            'n_train': len(y_train),
            'n_test': len(y_test),
            'R2': r2,
            'RMSE': rmse,
            'MAE': mae
        })

        # pooled prediction
        all_true.extend(y_test.tolist())
        all_pred.extend(pred.tolist())
        all_group.extend([g] * len(y_test))

        print(
            f'{name} | {str(g):<20} '
            f'n={len(y_test):<4} '
            f'R2={r2:>7.3f} '
            f'RMSE={rmse:>7.3f}'
        )

    # ========================================================
    # Result dataframe
    # ========================================================
    res_df = pd.DataFrame(results)

    if len(res_df) == 0:
        print('No valid groups.')
        return None

    res_df = res_df.sort_values('R2')

    # ========================================================
    # Overall metrics
    # ========================================================
    overall_r2   = r2_score(all_true, all_pred)
    overall_rmse = np.sqrt(mean_squared_error(all_true, all_pred))
    overall_mae  = mean_absolute_error(all_true, all_pred)

    summary = {
        'Validation': name,
        'Groups': len(res_df),

        'Mean_R2': res_df['R2'].mean(),
        'Median_R2': res_df['R2'].median(),

        'Positive_R2_%':
            (res_df['R2'] > 0).mean() * 100,

        'Overall_R2': overall_r2,
        'Overall_RMSE': overall_rmse,
        'Overall_MAE': overall_mae
    }

    # ========================================================
    # Save group metrics
    # ========================================================
    res_df.to_excel(
        f'../dataset/LOSO/{name}_results.xlsx',
        index=False
    )

    # ========================================================
    # Save predictions
    # ========================================================
    pred_df = pd.DataFrame({
        'Group': all_group,
        'Observed': all_true,
        'Predicted': all_pred,
        'Residual': np.array(all_true) - np.array(all_pred)
    })

    pred_df.to_excel(
        f'../../dataset/LOSO/{name}_predictions.xlsx',
        index=False
    )

    # ========================================================
    # Scatter plot
    # ========================================================
    plt.figure(figsize=(6, 6))

    sns.scatterplot(
        x=all_true,
        y=all_pred,
        s=60,
        alpha=0.7
    )

    lims = [
        min(min(all_true), min(all_pred)),
        max(max(all_true), max(all_pred))
    ]

    plt.plot(lims, lims, '--', lw=2)

    plt.xlim(lims)
    plt.ylim(lims)

    plt.xlabel('Observed lnBCF', fontsize=13)
    plt.ylabel('Predicted lnBCF', fontsize=13)

    plt.title(
        f'{name}: Overall Prediction',
        fontsize=14
    )

    plt.text(
        0.05,
        0.95,
        (
            f'Overall R² = {overall_r2:.3f}\n'
            f'RMSE = {overall_rmse:.3f}\n'
            f'MAE = {overall_mae:.3f}'
        ),
        transform=plt.gca().transAxes,
        va='top',
        fontsize=11
    )

    plt.tight_layout()

    plt.savefig(
        f'../../Fig/{name}_scatter.png',
        dpi=600,
        bbox_inches='tight'
    )

    plt.savefig(
        f'../../Fig/{name}_scatter.pdf',
        dpi=600,
        bbox_inches='tight'
    )

    plt.close()

    # ========================================================
    # Group-wise R2 plot
    # ========================================================
    plt.figure(figsize=(11, 6))

    colors = [
        'tomato' if x < 0 else 'steelblue'
        for x in res_df['R2']
    ]

    plt.bar(
        range(len(res_df)),
        res_df['R2'],
        color=colors
    )

    plt.axhline(0, ls='--', c='black')

    plt.xticks(
        range(len(res_df)),
        res_df['Group'],
        rotation=90,
        fontsize=8
    )

    plt.ylabel('R²', fontsize=12)

    plt.title(
        f'{name}: Group-wise Performance',
        fontsize=14
    )

    plt.tight_layout()

    plt.savefig(
        f'../../Fig/{name}_bar_R2.png',
        dpi=600,
        bbox_inches='tight'
    )

    plt.savefig(
        f'../../Fig/{name}_bar_R2.pdf',
        dpi=600,
        bbox_inches='tight'
    )

    plt.close()

    # ========================================================
    # Residual distribution
    # ========================================================
    plt.figure(figsize=(6, 4))

    residuals = np.array(all_true) - np.array(all_pred)

    sns.histplot(
        residuals,
        bins=30,
        kde=True
    )

    plt.axvline(0, ls='--', c='black')

    plt.xlabel('Residual')
    plt.ylabel('Count')

    plt.title(
        f'{name}: Residual Distribution',
        fontsize=14
    )

    plt.tight_layout()

    plt.savefig(
        f'../../Fig/{name}_residual.png',
        dpi=600,
        bbox_inches='tight'
    )

    plt.savefig(
        f'../../Fig/{name}_residual.pdf',
        dpi=600,
        bbox_inches='tight'
    )

    plt.close()

    # ========================================================
    # Print summary
    # ========================================================
    print('\n========== Summary ==========')

    for k, v in summary.items():

        if isinstance(v, float):
            print(f'{k:<20}: {v:.4f}')
        else:
            print(f'{k:<20}: {v}')

    return summary

In [19]:
summary_list = []

summary_list.append(run_group_cv(group_reference, name='LOSO'))


========== Running LOSO ==========
Total groups: 13
LOSO | 1                    n=32   R2=  0.706 RMSE=  0.835
LOSO | 2                    n=95   R2=  0.025 RMSE=  1.348
LOSO | 3                    n=58   R2= -0.558 RMSE=  2.649
LOSO | 4                    n=115  R2= -0.954 RMSE=  1.618
LOSO | 5                    n=43   R2=  0.521 RMSE=  1.497
LOSO | 6                    n=24   R2=  0.249 RMSE=  1.715
LOSO | 7                    n=168  R2= -0.598 RMSE=  1.821
LOSO | 8                    n=12   R2=  0.146 RMSE=  1.149
LOSO | 9                    n=128  R2=  0.360 RMSE=  1.466
LOSO | 10                   n=12   R2=  0.437 RMSE=  0.746
LOSO | 11                   n=18   R2= -1.797 RMSE=  1.479
LOSO | 12                   n=549  R2=  0.430 RMSE=  3.015
LOSO | 13                   n=98   R2=  0.550 RMSE=  1.564

========== Summary ==========
Validation          : LOSO
Groups              : 13
Mean_R2             : -0.0372
Median_R2           : 0.2492
Positive_R2_%       : 69.2308
Overall_

In [20]:
summary_list.append(run_group_cv(group_species,   name='LOSpeciesO'))


========== Running LOSpeciesO ==========
Total groups: 11
LOSpeciesO | lettuce              n=253  R2=  0.139 RMSE=  1.909
LOSpeciesO | tomato               n=69   R2=  0.595 RMSE=  1.216
LOSpeciesO | radish               n=185  R2=  0.401 RMSE=  2.255
LOSpeciesO | celery               n=20   R2=  0.498 RMSE=  0.831
LOSpeciesO | pea                  n=257  R2=  0.759 RMSE=  2.032
LOSpeciesO | pumpkin              n=58   R2= -0.558 RMSE=  2.649
LOSpeciesO | wheat                n=139  R2=  0.158 RMSE=  1.363
LOSpeciesO | maize                n=183  R2=  0.563 RMSE=  2.458
LOSpeciesO | red chicory          n=168  R2= -0.598 RMSE=  1.821
LOSpeciesO | soybean              n=12   R2=  0.146 RMSE=  1.149
LOSpeciesO | carrot               n=8    R2= -3.099 RMSE=  0.602

========== Summary ==========
Validation          : LOSpeciesO
Groups              : 11
Mean_R2             : -0.0907
Median_R2           : 0.1576
Positive_R2_%       : 72.7273
Overall_R2          : 0.5642
Overall_RMSE       

In [ ]:
summary_list.append(run_group_cv(group_pfas, name='LOPFASO'))


========== Running LOPFASO ==========
Total groups: 26
LOPFASO | PFBA                 n=120  R2=  0.493 RMSE=  1.558
LOPFASO | PFPeA                n=113  R2=  0.672 RMSE=  1.072
LOPFASO | PFHxA                n=122  R2=  0.484 RMSE=  1.044
LOPFASO | PFHpA                n=106  R2=  0.681 RMSE=  1.035
LOPFASO | PFOA                 n=126  R2=  0.743 RMSE=  1.079
LOPFASO | PFNA                 n=89   R2=  0.815 RMSE=  0.995
LOPFASO | PFDA                 n=91   R2=  0.815 RMSE=  0.960
LOPFASO | PFBS                 n=118  R2=  0.540 RMSE=  1.198
LOPFASO | PFHxS                n=48   R2=  0.849 RMSE=  0.556
Skip PFDS (n=1)
LOPFASO | PFOS                 n=135  R2=  0.767 RMSE=  1.085
LOPFASO | 6:2 diPAP            n=5    R2= -1.986 RMSE=  1.005
LOPFASO | 8:2 diPAP            n=5    R2= -0.102 RMSE=  1.382
Skip 8:2/10:2 diPAP (n=1)
LOPFASO | PFUnDA               n=57   R2=  0.875 RMSE=  0.992
LOPFASO | PFMOAA               n=16   R2=  0.740 RMSE=  0.637
LOPFASO | 4:2 FTS              n=1

In [ ]:
# ============================================================
# Final summary table
# ============================================================
summary_df = pd.DataFrame(summary_list)
summary_df.to_excel('../../dataset/LOSO/Generalization_Summary.xlsx', index=False)

print('\n================ Final Summary ================')
print(summary_df)


================ Final Summary ================
   Validation  Groups   Mean_R2  Median_R2  Positive_R2_%  Overall_R2  \
0        LOSO      13 -0.037182   0.249222      69.230769    0.417831   
1  LOSpeciesO      11 -0.090688   0.157630      72.727273    0.564162   
2     LOPFASO      21  0.446476   0.680771      80.952381    0.865691   

   Overall_RMSE  Overall_MAE  
0      2.306834     1.786914  
1      1.995970     1.541591  
2      1.105958     0.839721  


Hierarchical transferability across studies, species, and chemicals

Model transferability varied substantially across validation schemes. Prediction of unseen PFAS compounds remained robust (Overall R² = 0.864), indicating that chemical descriptors captured generalizable structure–uptake relationships. Transferability to unseen plant species was moderate (Overall R² = 0.623), whereas leave-one-study-out validation showed the largest performance decline (Overall R² = 0.471; mean study-level R² = -0.074), highlighting strong cross-study heterogeneity.

Although direct transfer across studies was limited, only a few local calibration samples substantially improved performance.